### Getting Common IDs across model scales

In [ ]:
import json
import pandas as pd
from pathlib import Path
from typing import Union, Optional


def write_common_unique_ids(
    path1: Union[str, Path],
    path2: Union[str, Path],
    output_path: Optional[Union[str, Path]] = None,
    unique_id_col: str = "unique_id",
    **read_json_kwargs,
) -> list:
    """
    Load two JSON/JSONL files into DataFrames, compute common unique_ids, and
    optionally write them to a JSON file.

    read_json_kwargs are passed to pd.read_json (e.g., lines=True).
    Returns the list of common ids.
    """
    path1, path2 = Path(path1), Path(path2)
    df1 = pd.read_json(path1, **read_json_kwargs)
    df2 = pd.read_json(path2, **read_json_kwargs)

    common_ids = sorted(set(df1[unique_id_col]) & set(df2[unique_id_col]))

    if output_path is not None:
        output_path = Path(output_path)
        output_path.write_text(json.dumps(common_ids, indent=2))

    return common_ids

subjects = ["math", "mmlu_math", "mmlu_physics"]

all_common = {}

for subject in subjects:
    path1 = f"../data/output_{subject}/activation_patching/qwen-1p5B/all_rfh_heads.jsonl"
    path2 = f"../data/output_{subject}/activation_patching/qwen-7B/all_rfh_heads.jsonl"

    common_ids = write_common_unique_ids(
        path1,
        path2,
        output_path=None,   # don't write per-subject; we want one combined file
        lines=True,
    )
    all_common[subject] = common_ids
    print(f"{subject}: {len(common_ids)} common ids")

# write the combined mapping
output_path = Path("../data/common_unique_ids.json")
with open(output_path, "w") as f:
    json.dump(all_common, f, indent=2)

print(f"wrote common ids for {len(subjects)} subjects to {output_path}")


### Metrics for all-RFH Patching

In [115]:
import os
import json
import pandas as pd
from pathlib import Path

with open("../data/common_unique_ids.json", "r") as f:
    ids = json.load(f) 

subject_to_dataset = {
    "math": "MATH-500",
    "mmlu_math": "MMLU-math",
    "mmlu_physics": "MMLU-physics"
}
all_rfh_df = []
subjects = ["math", "mmlu_math", "mmlu_physics"]
models = ["qwen-1p5B", "qwen-7B"]
for subject in subjects:
    for model in models:
        path = os.path.join(os.getcwd(), f"../data/output_{subject}/activation_patching/{model}/all_rfh_heads.jsonl")
        try:
            temp = pd.read_json(path, lines = True)
            temp.drop_duplicates(subset="unique_id", inplace=True)
            temp["dataset"] = subject_to_dataset[subject]
            temp = temp[temp["unique_id"].isin(ids[subject])]
            all_rfh_df.append(temp)
        except:
            print(path)
            continue
all_rfh_df = pd.concat(all_rfh_df, ignore_index=True)

In [129]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def plot_all_rfh_mediated_fraction(all_rfh_df: pd.DataFrame,
                                   outfile: str = "all_rfh_mediated_fraction.pdf") -> None:
    """
    Global RFH-mediated fraction of CoT effect (NIE / TE, %) by model and dataset.

    Assumes all_rfh_df has columns:
        - 'model_name'
        - 'dataset'
        - 'withR_loss'             # CoT = 1, RFH = 1   (World A)
        - 'withoutR_loss'          # CoT = 0, RFH = 0   (World B)
        - 'patched_withoutR_loss'  # CoT = 0, RFH = 1   (World D)
        - 'patched_withR_loss'     # CoT = 1, RFH = 0   (World C)

    We compute (on mean CE loss):
        TE  = B - A
        NIE = B - D
        NDE = B - C
        NIE/TE is then converted to a percentage.
    """

    # ---------- Aggregation and metric computation ----------
    grouped = (
        all_rfh_df
        .groupby(["dataset", "model_name"], as_index=False)[
            ["withR_loss", "withoutR_loss", "patched_withoutR_loss", "patched_withR_loss"]
        ]
        .mean()
    )

    # Rename for clarity
    grouped = grouped.rename(columns={
        "withR_loss": "A",                 # CoT = 1, RFH = 1
        "withoutR_loss": "B",              # CoT = 0, RFH = 0
        "patched_withR_loss": "C",         # CoT = 1, RFH = 0
        "patched_withoutR_loss": "D"       # CoT = 0, RFH = 1
    })

    grouped["TE"] = grouped["B"] - grouped["A"]
    grouped["NIE"] = grouped["B"] - grouped["D"]
    grouped["NDE"] = grouped["B"] - grouped["C"]

    eps = 1e-8
    grouped["NIE_over_TE"] = np.where(
        np.abs(grouped["TE"]) > eps,
        grouped["NIE"] / grouped["TE"],
        np.nan
    )
    grouped["NIE_over_TE_pct"] = 100.0 * grouped["NIE_over_TE"]

    # ---------- Prepare plotting structures ----------
    datasets = sorted(grouped["dataset"].unique())
    models = sorted(grouped["model_name"].unique())

    # Pretty mapping for model labels
    model_label_map = {
        "qwen-1p5B": "Qwen-1.5B",
        "qwen-7B": "Qwen-7B",
    }
    model_labels = [model_label_map.get(m, m) for m in models]

    # ---------- Matplotlib style for paper-quality figure ----------
    plt.rcParams.update({
        "font.size": 10,
        "axes.labelsize": 10,
        "axes.titlesize": 10,
        "xtick.labelsize": 9,
        "ytick.labelsize": 9,
        "legend.fontsize": 9,
        "figure.dpi": 300,
    })

    fig, axes = plt.subplots(
        1, len(datasets),
        figsize=(3.3 * len(datasets), 2.9),
        sharey=True
    )

    if len(datasets) == 1:
        axes = [axes]

    width = 0.6  # bar width

    # Colors by model (consistent across subplots)
    from matplotlib.cm import get_cmap
    cmap = get_cmap("tab10")
    colors = [cmap(i) for i in range(len(models))]

    bar_handles = None
    global_min, global_max = [], []

    panel_labels = ["(a)", "(b)", "(c)", "(d)"]

    for ax_idx, (ax, ds) in enumerate(zip(axes, datasets)):
        ds_data = grouped[grouped["dataset"] == ds]

        heights = []
        for m in models:
            val = ds_data.loc[ds_data["model_name"] == m, "NIE_over_TE_pct"]
            heights.append(float(val.iloc[0]) if not val.empty else np.nan)

        x = np.arange(len(models))

        bars = ax.bar(
            x,
            heights,
            width=width,
            color=[colors[i] for i in range(len(models))],
            edgecolor="black",
            linewidth=0.6
        )

        if bar_handles is None:
            bar_handles = bars

        # Cosmetic tweaks
        ax.axhline(0.0, color="black", linewidth=0.8, alpha=0.9)
        ax.grid(axis="y", linestyle=":", linewidth=0.6, alpha=0.7)

        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

        ax.set_xticks(x)
        ax.set_xticklabels(model_labels, rotation=0)
        ax.set_title(ds, loc="left")

        finite_heights = [h for h in heights if not np.isnan(h)]
        if finite_heights:
            global_min.append(min(finite_heights))
            global_max.append(max(finite_heights))

        # Only leftmost subplot gets y-axis label
        if ax is axes[0]:
            ax.set_ylabel("(NIE/TE) (%)")

        # Panel label (a), (b), ...
        ax.text(
            0.02, 0.94,
            panel_labels[ax_idx],
            transform=ax.transAxes,
            fontsize=10,
            fontweight="bold",
            va="top",
            ha="left"
        )

    # Shared y-limits across datasets for comparability
    if global_min and global_max:
        ymin = min(0.0, min(global_min) - 5)
        ymax = max(0.0, max(global_max) + 5)
        for ax in axes:
            ax.set_ylim(ymin, ymax)

    # Common x-axis label
    fig.text(0.5, 0.01, "Model", ha="center")

    # Shared legend (models)
    fig.legend(
        bar_handles,
        model_labels,
        loc="upper center",
        bbox_to_anchor=(0.5, 1.07),
        ncol=len(models),
        frameon=False
    )

    fig.tight_layout(rect=[0.03, 0.03, 0.98, 0.92])
    fig.savefig(outfile, bbox_inches="tight")
    plt.close(fig)


plot_all_rfh_mediated_fraction(all_rfh_df, outfile="all_rfh_mediated_fraction.pdf")

/var/folders/sn/px3mjtjs55dcc7gkvsfb0knw0000gn/T/ipykernel_19970/2132798244.py:89: MatplotlibDeprecationWarning:

The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.



## Metrics for Layer-wise patching

In [119]:
import os
import pandas as pd
from pathlib import Path

subject_to_dataset = {
    "math": "MATH-500",
    "mmlu_math": "MMLU-math",
    "mmlu_physics": "MMLU-physics"
}

layer_map = {
    "qwen-1p5B": [1, 12, 14, 16, 19, 20, 23],
    "qwen-7B": [16, 19, 14, 1, 17, 22]
}
layerwise_df = []
subjects = ["math", "mmlu_math", "mmlu_physics"]
models = ["qwen-1p5B", "qwen-7B"]
for subject in subjects:
    for model in models:
        layers = layer_map[model]
        for layer in layers:
            path = f"../data/output_{subject}/activation_patching/{model}/layer_{layer}_rfh_heads.jsonl"
            try:
                temp = pd.read_json(path, lines = True)
                temp.drop_duplicates(subset="unique_id", inplace=True)
                temp["dataset"] = subject_to_dataset[subject]
                temp["layer"] = layer
                temp = temp[temp["unique_id"].isin(ids[subject])]
                layerwise_df.append(temp)
            except:
                print(path)
                continue
layerwise_df = pd.concat(layerwise_df, ignore_index=True)

In [138]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def plot_layerwise_mediated_fraction(layerwise_df: pd.DataFrame,
                                     outfile: str = "layerwise_rfh_mediated_fraction.pdf") -> None:
    """
    Layer-wise RFH-mediated fraction of CoT effect (NIE / TE, %) vs. layer index.
    One subplot per dataset (stacked vertically), lines colored / styled by model.
    """

    # ---------- Aggregation and metric computation ----------
    grouped = (
        layerwise_df
        .groupby(["dataset", "layer", "model_name"], as_index=False)[
            ["withR_loss", "withoutR_loss", "patched_withoutR_loss", "patched_withR_loss"]
        ]
        .mean()
    )

    grouped["layer"] = grouped["layer"].astype(int)

    grouped = grouped.rename(columns={
        "withR_loss": "A",                 # CoT = 1, RFH = 1
        "withoutR_loss": "B",              # CoT = 0, RFH = 0
        "patched_withR_loss": "C",         # CoT = 1, RFH = 0
        "patched_withoutR_loss": "D"       # CoT = 0, RFH = 1
    })

    grouped["TE"] = grouped["B"] - grouped["A"]
    grouped["NIE"] = grouped["B"] - grouped["D"]
    grouped["NDE"] = grouped["B"] - grouped["C"]

    eps = 1e-8
    grouped["NIE_over_TE"] = np.where(
        np.abs(grouped["TE"]) > eps,
        grouped["NIE"] / grouped["TE"],
        np.nan
    )
    grouped["NIE_over_TE_pct"] = 100.0 * grouped["NIE_over_TE"]

    datasets = sorted(grouped["dataset"].unique())
    models = sorted(grouped["model_name"].unique())

    model_label_map = {
        "qwen-1p5B": "Qwen-1.5B",
        "qwen-7B": "Qwen-7B",
    }
    model_labels = [model_label_map.get(m, m) for m in models]

    # ---------- Matplotlib style ----------
    plt.rcParams.update({
        "font.size": 10,
        "axes.labelsize": 10,
        "axes.titlesize": 10,
        "xtick.labelsize": 9,
        "ytick.labelsize": 9,
        "legend.fontsize": 9,
        "figure.dpi": 300,
    })

    fig, axes = plt.subplots(
        len(datasets), 1,
        figsize=(4.0, 2.6 * len(datasets)),
        sharex=True,
        sharey=False
    )
    if len(datasets) == 1:
        axes = [axes]

    from matplotlib.cm import get_cmap
    cmap = get_cmap("tab10")
    colors = [cmap(i) for i in range(len(models))]

    # differentiate lines by marker and linestyle as well as color
    markers = ["o", "^", "s", "D", "v"]
    linestyles = ["-", "-", "-.", ":"]

    model_handles = {}
    panel_labels = ["(a)", "(b)", "(c)", "(d)", "(e)"]  # enough for 3+ datasets

    for ax_idx, (ax, ds) in enumerate(zip(axes, datasets)):
        ds_data = grouped[grouped["dataset"] == ds]

        all_heights = []

        for j, (m, label) in enumerate(zip(models, model_labels)):
            sub = ds_data[ds_data["model_name"] == m].sort_values("layer")
            if sub.empty:
                continue

            x = sub["layer"].values
            y = sub["NIE_over_TE_pct"].values
            all_heights.extend([v for v in y if not np.isnan(v)])

            color = colors[j % len(colors)]
            marker = markers[j % len(markers)]
            linestyle = linestyles[j % len(linestyles)]

            (line,) = ax.plot(
                x, y,
                marker=marker,
                linestyle=linestyle,
                linewidth=1.4,
                markersize=3.2,
                color=color,
                label=label
            )

            if m not in model_handles:
                model_handles[m] = line

        # Cosmetic tweaks
        ax.axhline(0.0, color="black", linewidth=0.8, alpha=0.8)
        ax.grid(axis="y", linestyle=":", linewidth=0.6, alpha=0.7)

        # Remove top/right spines for a cleaner look
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

        # Y-axis limits per subplot
        finite_heights = [h for h in all_heights if not np.isnan(h)]
        if finite_heights:
            y_min = min(0.0, min(finite_heights) - 5)
            y_max = max(0.0, max(finite_heights) + 5)
            ax.set_ylim(y_min, y_max)

        # Title and panel label
        ax.set_title(ds, loc="left")
        ax.text(
            0.01, 0.92,
            panel_labels[ax_idx],
            transform=ax.transAxes,
            fontsize=10,
            fontweight="bold",
            va="top",
            ha="left"
        )

        ax.set_ylabel("NIE/TE (%)")

    axes[-1].set_xlabel("Layer index")

    handles = [model_handles[m] for m in models if m in model_handles]
    labels = [model_label_map.get(m, m) for m in models if m in model_handles]

    fig.legend(
        handles,
        labels,
        loc="upper center",
        bbox_to_anchor=(0.5, 0.995),
        ncol=len(handles),
        frameon=False
    )

    fig.tight_layout(rect=[0.08, 0.04, 0.98, 0.93])
    fig.savefig(outfile, bbox_inches="tight")
    plt.close(fig)

plot_layerwise_mediated_fraction(layerwise_df)

/var/folders/sn/px3mjtjs55dcc7gkvsfb0knw0000gn/T/ipykernel_19970/3117000551.py:72: MatplotlibDeprecationWarning:

The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.



### Metrics for Top-k Patching

In [131]:
import pandas as pd

topk_df = []
subjects = ["math", "mmlu_math", "mmlu_physics"]
models = ["qwen-1p5B", "qwen-7B"]
for subject in subjects:
    for model in models:
        for k in range(1, 6):
            path = f"../data/output_{subject}/activation_patching/{model}/top_{k}_rfh.jsonl"
            try:
                temp = pd.read_json(path, lines = True)
                temp.drop_duplicates(subset="unique_id", inplace=True)
                temp["dataset"] = subject_to_dataset[subject]
                temp["top_k"] = k
                temp = temp[temp["unique_id"].isin(ids[subject])]
                topk_df.append(temp)
            except:
                print(path)
                continue
topk_df = pd.concat(topk_df, ignore_index=True)

In [137]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def plot_topk_suff_necess(topk_df: pd.DataFrame,
                          outfile: str = "topk_rfh_suff_necess.pdf") -> None:
    """
    Top-k sufficiency and necessity of RFHs.

    For each (dataset, model_name, top_k), we compute:
        TE          = withoutR_loss - withR_loss
        NIE/TE      = (withoutR_loss - patched_withoutR_loss) / TE
        NDE_swap/TE = (withoutR_loss - patched_withR_loss) / TE
        NDE_ablt/TE = (withoutR_loss - ablated_withR_loss) / TE

    We plot:
      - Row 1 (sufficiency): NIE/TE (%) vs top_k
      - Row 2 (necessity):  NDE/TE (%) vs top_k (swap and ablate)
    Columns correspond to datasets.
    """

    # --------- Aggregation and metric computation ----------
    grouped = (
        topk_df
        .groupby(["dataset", "top_k", "model_name"], as_index=False)[
            [
                "withR_loss",
                "withoutR_loss",
                "patched_withoutR_loss",
                "patched_withR_loss",
                "ablated_withR_loss",
            ]
        ]
        .mean()
    )

    grouped["top_k"] = grouped["top_k"].astype(int)

    grouped = grouped.rename(columns={
        "withR_loss": "A",                 # CoT = 1, RFH = full
        "withoutR_loss": "B",              # CoT = 0, RFH = none
        "patched_withoutR_loss": "D",      # CoT = 0, RFH = top-k from CoT
        "patched_withR_loss": "C",         # CoT = 1, RFH = top-k from no-CoT
        "ablated_withR_loss": "E",         # CoT = 1, RFH = top-k ablated
    })

    # Total effect
    grouped["TE"] = grouped["B"] - grouped["A"]

    # Numerators
    grouped["NIE_num"] = grouped["B"] - grouped["D"]
    grouped["NDE_swap_num"] = grouped["B"] - grouped["C"]
    grouped["NDE_ablate_num"] = grouped["B"] - grouped["E"]

    eps = 1e-8
    safe_TE = np.where(np.abs(grouped["TE"]) > eps, grouped["TE"], np.nan)

    grouped["NIE_over_TE"] = grouped["NIE_num"] / safe_TE
    grouped["NDE_swap_over_TE"] = grouped["NDE_swap_num"] / safe_TE
    grouped["NDE_ablate_over_TE"] = grouped["NDE_ablate_num"] / safe_TE

    grouped["NIE_over_TE_pct"] = 100.0 * grouped["NIE_over_TE"]
    grouped["NDE_swap_over_TE_pct"] = 100.0 * grouped["NDE_swap_over_TE"]
    grouped["NDE_ablate_over_TE_pct"] = 100.0 * grouped["NDE_ablate_over_TE"]

    datasets = sorted(grouped["dataset"].unique())
    models = sorted(grouped["model_name"].unique())

    model_label_map = {
        "qwen-1p5B": "Qwen-1.5B",
        "qwen-7B": "Qwen-7B",
    }
    model_labels = [model_label_map.get(m, m) for m in models]

    # --------- Matplotlib style (camera-ready) ----------
    plt.rcParams.update({
        "font.size": 10,
        "axes.labelsize": 10,
        "axes.titlesize": 10,
        "xtick.labelsize": 9,
        "ytick.labelsize": 9,
        "legend.fontsize": 9,
        "figure.dpi": 300,
    })

    n_datasets = len(datasets)

    # 2 rows (sufficiency / necessity) x columns = datasets
    fig, axes = plt.subplots(
        2, n_datasets,
        figsize=(3.2 * n_datasets, 4.6),
        sharex=True,
        sharey="row",
    )

    if n_datasets == 1:
        # Ensure 2D indexing works even with one dataset
        axes = np.array(axes).reshape(2, 1)

    from matplotlib.cm import get_cmap
    cmap = get_cmap("tab10")
    colors = [cmap(i) for i in range(len(models))]
    markers = ["o", "^", "s", "D", "v"]

    # For global legends
    suff_handles = {}
    nec_handles = {}

    # For global y-limits per row
    suff_vals, nec_vals = [], []

    panel_labels = ["(a)", "(b)", "(c)", "(d)", "(e)", "(f)"]

    for col_idx, ds in enumerate(datasets):
        ds_data = grouped[grouped["dataset"] == ds]

        # Determine common k-values for xticks
        k_values = sorted(ds_data["top_k"].unique())

        ax_suff = axes[0, col_idx]  # row 0: sufficiency
        ax_nec = axes[1, col_idx]   # row 1: necessity

        # ---------- Row 0: Sufficiency (NIE/TE) ----------
        for j, (m, m_label) in enumerate(zip(models, model_labels)):
            sub = ds_data[ds_data["model_name"] == m].sort_values("top_k")
            if sub.empty:
                continue

            x = sub["top_k"].values
            y = sub["NIE_over_TE_pct"].values
            suff_vals.extend([v for v in y if not np.isnan(v)])

            color = colors[j % len(colors)]
            marker = markers[j % len(markers)]

            (line,) = ax_suff.plot(
                x, y,
                marker=marker,
                linestyle="-",
                linewidth=1.4,
                markersize=3.2,
                color=color,
                label=m_label,
            )

            if m not in suff_handles:
                suff_handles[m] = line

        ax_suff.axhline(0.0, color="black", linewidth=0.8, alpha=0.9)
        ax_suff.grid(axis="y", linestyle=":", linewidth=0.6, alpha=0.7)
        ax_suff.spines["top"].set_visible(False)
        ax_suff.spines["right"].set_visible(False)
        ax_suff.set_xticks(k_values)
        ax_suff.set_xlim(min(k_values) - 0.2, max(k_values) + 0.2)

        # Dataset title on top row
        ax_suff.set_title(ds)

        # Y-axis label only on leftmost subplot in row 0
        if col_idx == 0:
            ax_suff.set_ylabel("NIE/TE (%)")

        # Panel label (a), (b), ...
        ax_suff.text(
            0.02, 0.94,
            panel_labels[col_idx * 2],
            transform=ax_suff.transAxes,
            fontsize=10,
            fontweight="bold",
            va="top",
            ha="left",
        )

        # ---------- Row 1: Necessity (NDE/TE, swap + ablate) ----------
        for j, (m, m_label) in enumerate(zip(models, model_labels)):
            sub = ds_data[ds_data["model_name"] == m].sort_values("top_k")
            if sub.empty:
                continue

            x = sub["top_k"].values
            y_swap = sub["NDE_swap_over_TE_pct"].values
            y_ablt = sub["NDE_ablate_over_TE_pct"].values

            nec_vals.extend([v for v in y_swap if not np.isnan(v)])
            nec_vals.extend([v for v in y_ablt if not np.isnan(v)])

            color = colors[j % len(colors)]
            marker = markers[j % len(markers)]

            # swap line
            (line_swap,) = ax_nec.plot(
                x, y_swap,
                marker=marker,
                linestyle="-",
                linewidth=1.4,
                markersize=3.0,
                color=color,
                label=f"{m_label} (swap)",
            )
            # ablate line
            (line_ablt,) = ax_nec.plot(
                x, y_ablt,
                marker=marker,
                linestyle="--",
                linewidth=1.4,
                markersize=3.0,
                color=color,
                label=f"{m_label} (ablate)",
            )

            # Just grab one handle per label for the legend (first dataset only)
            if col_idx == 0:
                nec_handles[f"{m}_swap"] = line_swap
                nec_handles[f"{m}_ablate"] = line_ablt

        ax_nec.axhline(0.0, color="black", linewidth=0.8, alpha=0.9)
        ax_nec.grid(axis="y", linestyle=":", linewidth=0.6, alpha=0.7)
        ax_nec.spines["top"].set_visible(False)
        ax_nec.spines["right"].set_visible(False)
        ax_nec.set_xticks(k_values)
        ax_nec.set_xlim(min(k_values) - 0.2, max(k_values) + 0.2)

        # X labels on bottom row only
        ax_nec.set_xlabel("Top-k RFHs")

        # Y-axis label only on leftmost subplot in row 1
        if col_idx == 0:
            ax_nec.set_ylabel("NDE/TE (%)")

        # Panel label (next letter)
        ax_nec.text(
            0.02, 0.94,
            panel_labels[col_idx * 2 + 1],
            transform=ax_nec.transAxes,
            fontsize=10,
            fontweight="bold",
            va="top",
            ha="left",
        )

    # Global y-limits per row
    if suff_vals:
        ymin_s = min(0.0, min(suff_vals) - 5)
        ymax_s = max(0.0, max(suff_vals) + 5)
        for col_idx in range(n_datasets):
            axes[0, col_idx].set_ylim(ymin_s, ymax_s)

    if nec_vals:
        ymin_n = min(0.0, min(nec_vals) - 5)
        ymax_n = max(0.0, max(nec_vals) + 5)
        for col_idx in range(n_datasets):
            axes[1, col_idx].set_ylim(ymin_n, ymax_n)

    # --------- Legends grouped into boxes ---------
    # Sufficiency legend (models) – top center
    suff_leg_handles = [suff_handles[m] for m in models if m in suff_handles]
    suff_leg_labels = [model_label_map.get(m, m) for m in models if m in suff_handles]

    leg1 = fig.legend(
        suff_leg_handles,
        suff_leg_labels,
        loc="upper center",
        bbox_to_anchor=(0.5, 1.03),
        ncol=len(suff_leg_handles),
        frameon=True,
        fancybox=True,
        borderaxespad=0.4,
        title="Models (color/marker)",
    )

    # Necessity legend (model + swap/ablate) – bottom center
    nec_leg_handles = list(nec_handles.values())
    nec_leg_labels = [h.get_label() for h in nec_leg_handles]

    leg2 = fig.legend(
        nec_leg_handles,
        nec_leg_labels,
        loc="lower center",
        bbox_to_anchor=(0.5, -0.02),
        ncol=min(len(nec_leg_handles), 4),
        frameon=True,
        fancybox=True,
        borderaxespad=0.4,
        title="Intervention (swap vs. ablate)",
    )

    leg1.get_frame().set_linewidth(0.6)
    leg2.get_frame().set_linewidth(0.6)

    fig.tight_layout(rect=[0.06, 0.08, 0.98, 0.92])
    fig.savefig(outfile, bbox_inches="tight")
    plt.close(fig)


# Example usage:
plot_topk_suff_necess(topk_df, outfile="topk_rfh_suff_necess.pdf")

/var/folders/sn/px3mjtjs55dcc7gkvsfb0knw0000gn/T/ipykernel_19970/3617233199.py:101: MatplotlibDeprecationWarning:

The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.



### NDE/TE with induction, retrieval and RFH

In [147]:
import pandas as pd

irr_df = []
models = ["qwen-1p5B", "qwen-7B"]
files = [("all_induction_heads", "I"), ("all_induction_and_retrieval_heads", "IR"), ("all_induction_retrieval_and_rfh_heads", "IRR")]
for model in models:
    for file, code in files:
        path = f"../data/output_math/activation_patching/{model}/{file}.jsonl"
        try:
            temp = pd.read_json(path, lines = True)
            temp.drop_duplicates(subset="unique_id", inplace=True)
            temp["dataset"] = subject_to_dataset[subject]
            temp["category"] = code
            temp = temp[temp["unique_id"].isin(ids["math"])]
            irr_df.append(temp)
        except:
            print(path)
            continue
irr_df = pd.concat(irr_df, ignore_index=True)

In [157]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def plot_extended_circuit_nde(ext_df: pd.DataFrame,
                              outfile: str = "extended_circuit_nde.pdf"):
    """
    Extended ablation study over induction / retrieval / RFH heads.

    Assumes ext_df has columns:
        - 'model_name'
        - 'dataset'
        - 'category' in {"I", "IR", "IRR"}
        - 'withR_loss'             # CoT = 1, full circuit active
        - 'withoutR_loss'          # CoT = 0, full circuit inactive
        - 'patched_withR_loss'     # CoT = 1, circuit swapped to no-CoT (swap)
        - 'ablated_withR_loss'     # CoT = 1, circuit zeroed (ablate)

    For each (model_name, category) within the chosen dataset:
        TE          = withoutR_loss - withR_loss
        NDE_swap/TE = (withoutR_loss - patched_withR_loss) / TE
        NDE_ablt/TE = (withoutR_loss - ablated_withR_loss) / TE

    We plot:
        X-axis: head configuration (expanded labels for I, IR, IRR)
        Y-axis: NDE/TE (%) for swap and ablate, two lines per model.
    """

    df = ext_df

    # --------- Aggregation and metric computation ----------
    grouped = (
        df.groupby(["category", "model_name"], as_index=False)[
            [
                "withR_loss",
                "withoutR_loss",
                "patched_withR_loss",
                "ablated_withR_loss",
            ]
        ]
        .mean()
    )

    grouped = grouped.rename(columns={
        "withR_loss": "A",             # CoT = 1, full circuit
        "withoutR_loss": "B",          # CoT = 0, full circuit
        "patched_withR_loss": "C",     # CoT = 1, circuit swapped to no-CoT
        "ablated_withR_loss": "E",     # CoT = 1, circuit ablated (0)
    })

    # Total effect
    grouped["TE"] = grouped["B"] - grouped["A"]

    # NDE numerators
    grouped["NDE_swap_num"] = grouped["B"] - grouped["C"]
    grouped["NDE_ablate_num"] = grouped["B"] - grouped["E"]

    eps = 1e-8
    safe_TE = np.where(np.abs(grouped["TE"]) > eps, grouped["TE"], np.nan)

    grouped["NDE_swap_over_TE"] = grouped["NDE_swap_num"] / safe_TE
    grouped["NDE_ablate_over_TE"] = grouped["NDE_ablate_num"] / safe_TE

    grouped["NDE_swap_over_TE_pct"] = 100.0 * grouped["NDE_swap_over_TE"]
    grouped["NDE_ablate_over_TE_pct"] = 100.0 * grouped["NDE_ablate_over_TE"]

    # --------- Plotting setup ----------
    # Fix category order for x-axis
    cat_order = ["I", "IR", "IRR"]
    grouped["category"] = pd.Categorical(
        grouped["category"], categories=cat_order, ordered=True
    )

    models = sorted(grouped["model_name"].unique())

    model_label_map = {
        "qwen-1p5B": "Qwen-1.5B",
        "qwen-7B": "Qwen-7B",
    }
    model_labels = {m: model_label_map.get(m, m) for m in models}

    # Expanded labels for the x-axis
    cat_label_map = {
        "I": "Induction heads",
        "IR": "Induction + retrieval",
        "IRR": "Induction + retrieval + RFH",
    }

    # Camera-ready style
    plt.rcParams.update({
        "font.size": 10,
        "axes.labelsize": 10,
        "axes.titlesize": 10,
        "xtick.labelsize": 9,
        "ytick.labelsize": 9,
        "legend.fontsize": 9,
        "figure.dpi": 300,
    })

    fig, ax = plt.subplots(figsize=(5.4, 3.0))

    from matplotlib.cm import get_cmap
    cmap = get_cmap("tab10")
    colors = [cmap(i) for i in range(len(models))]
    markers = ["o", "^", "s", "D", "v"]

    # Map categories to x positions
    x_positions = {cat: i for i, cat in enumerate(cat_order)}
    x_vals = np.array(list(x_positions.values()))
    ax.set_xticks(x_vals)
    ax.set_xticklabels([cat_label_map[c] for c in cat_order], rotation=15, ha="right")

    nde_vals = []
    handles = []

    # One pair of lines per model
    for j, m in enumerate(models):
        sub = grouped[grouped["model_name"] == m].sort_values("category")
        if sub.empty:
            continue

        # Ensure order I, IR, IRR
        sub = sub.set_index("category").reindex(cat_order)

        x = np.array([x_positions[c] for c in cat_order])
        y_swap = sub["NDE_swap_over_TE_pct"].values
        y_ablt = sub["NDE_ablate_over_TE_pct"].values

        nde_vals.extend([v for v in y_swap if not np.isnan(v)])
        nde_vals.extend([v for v in y_ablt if not np.isnan(v)])

        color = colors[j % len(colors)]
        marker = markers[j % len(markers)]

        # Swap line
        (line_swap,) = ax.plot(
            x, y_swap,
            marker=marker,
            linestyle="-",
            linewidth=1.4,
            markersize=3.2,
            color=color,
            label=f"{model_labels[m]} (swap)",
        )
        # Ablate line
        (line_ablt,) = ax.plot(
            x, y_ablt,
            marker=marker,
            linestyle="--",
            linewidth=1.4,
            markersize=3.2,
            color=color,
            label=f"{model_labels[m]} (ablate)",
        )

        handles.extend([line_swap, line_ablt])

    # Cosmetic tweaks
    ax.axhline(0.0, color="black", linewidth=0.8, alpha=0.9)
    ax.grid(axis="y", linestyle=":", linewidth=0.6, alpha=0.7)

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    # Y-limits from data with a bit of margin
    if nde_vals:
        ymin = min(0.0, min(nde_vals) - 5)
        ymax = max(0.0, max(nde_vals) + 5)
        ax.set_ylim(ymin, ymax)

    ax.set_xlabel("Head configuration")
    ax.set_ylabel("NDE/TE (%)")

    # Legend outside the plot, centered above
    legend = fig.legend(
        handles,
        [h.get_label() for h in handles],
        loc="upper center",
        bbox_to_anchor=(0.5, 1.10),
        ncol=min(len(handles), 4),
        frameon=True,
        fancybox=True,
        borderaxespad=0.4,
        title="Model × intervention",
    )
    legend.get_frame().set_linewidth(0.6)

    # Leave space at top for the legend
    fig.tight_layout(rect=[0.06, 0.05, 0.98, 0.90])
    fig.savefig(outfile, bbox_inches="tight")
    plt.close(fig)


# Example usage:
plot_extended_circuit_nde(irr_df, outfile="extended_circuit_nde.pdf")

/var/folders/sn/px3mjtjs55dcc7gkvsfb0knw0000gn/T/ipykernel_19970/894022765.py:103: MatplotlibDeprecationWarning:

The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.

